In [1]:
import numpy as np
import scipy as sp

In [14]:
def mc_european(S0, K, T, r, sigma, M):
    """
    Monte Carlo pricer for European Call/Put options
    """

    rng = np.random.default_rng(42)

    random_array = rng.standard_normal(M)

    S_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + random_array * sigma * np.sqrt(T) )

    discount = np.exp( - r * T )
    
    c_array = discount * np.maximum( S_array[:, np.newaxis] - K , 0 )
    p_array = discount * np.maximum( K - S_array[:, np.newaxis] , 0 )

    c_se = np.std( c_array, axis=0, ddof=1 ) / np.sqrt( M )
    p_se = np.std( p_array, axis=0, ddof=1 ) / np.sqrt( M )

    return c_array.mean( axis=0 ), p_array.mean( axis=0 ), c_se, p_se

In [ ]:
S0 = 100.0
K = np.array([90, 100, 110]) 
T = 1.0
r = 0.05
sigma = 0.20
M = 1000000

c_naive, p_naive, c_se_naive, p_se_naive = mc_european(S0, K, T, r, sigma, M)
print("Calls:", c_naive, "SE:", c_se_naive)
print("Puts:", p_naive, "SE:", p_se_naive)

Calls: [16.70336868 10.45319528  6.0440394 ] SE: [0.01739816 0.01473146 0.01164752]
Puts: [ 2.3102142   5.57233504 10.6754734 ] SE: [0.00535468 0.00866209 0.0119931 ]


In [15]:
def mc_european_antithetic(S0, K, T, r, sigma, M):
    """
    Monte Carlo pricer for European Call/Put options using antithetic variates
    """

    rng = np.random.default_rng(42)
    
    random_array = rng.standard_normal( int( M / 2 ))
    neg_random_array = ( - 1 ) * random_array

    S1_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + random_array * sigma * np.sqrt(T) )
    S2_array = S0 * np.exp( ( r - sigma ** 2 / 2 ) * T + neg_random_array * sigma * np.sqrt(T) )

    discount = np.exp( - r * T )

    c1_array = discount * np.maximum( S1_array[:, np.newaxis] - K , 0 )
    c2_array = discount * np.maximum( S2_array[:, np.newaxis] - K , 0 )
    p1_array = discount * np.maximum( K - S1_array[:, np.newaxis] , 0 )
    p2_array = discount * np.maximum( K - S2_array[:, np.newaxis] , 0 )

    c_array = ( c1_array + c2_array ) / 2
    p_array = ( p1_array + p2_array ) / 2

    c_se = np.std( c_array, axis=0, ddof=1 ) / np.sqrt( M / 2 )
    p_se = np.std( p_array, axis=0, ddof=1 ) / np.sqrt( M / 2 )

    return c_array.mean( axis=0 ), p_array.mean( axis=0 ), c_se, p_se

In [16]:
S0 = 100.0
K = np.array([90, 100, 110]) 
T = 1.0
r = 0.05
sigma = 0.20
M = 1000000

c_anti, p_anti, c_se_anti, p_se_anti = mc_european_antithetic(S0, K, T, r, sigma, M)
print("Calls:", c_anti, "SE:", c_se_anti)
print("Puts:", p_anti, "SE:", p_se_anti)

Calls: [16.70598923 10.45653567  6.04596594] SE: [0.00876726 0.01041622 0.00996302]
Puts: [ 2.31340091  5.5762416  10.67796611] SE: [0.00482894 0.00663198 0.00611813]


In [24]:
## VARIANCE REDUCTION RATIO (VRR)

print("VRR for calls:", ( c_se_naive / c_se_anti ) ** 2)
print("VRR for puts:", ( p_se_naive / p_se_anti ) ** 2)

VRR for calls: [3.93802415 2.00018912 1.3667361 ]
VRR for puts: [1.22959658 1.70592081 3.84259906]
